# PartCrafter on Kaggle — image -> parts GLB (robust build)

**REQUIRED (notebook can't set these for you):**
- Right panel **Settings -> Accelerator = GPU T4 x2**
- Right panel **Settings -> Internet = On**  (needed for git clone + pip + model download)

Then **Run All**. The input image is embedded (no upload). At the end a `.glb` appears
in the **Output** panel. Cell 2 takes ~15 min (mostly silent except [1/5]..[5/5] markers).

In [ ]:
# 1) GPU check (must show Tesla T4). If this errors -> GPU not enabled in Settings.
!nvidia-smi


In [ ]:
# 2) Install PartCrafter (~15 min). Progress is printed between steps.
import os, time, subprocess, sys
t0 = time.time()
os.chdir('/kaggle/working')
if not os.path.isdir('PartCrafter/.git'):
    rc = os.system('git clone https://github.com/wgsxm/PartCrafter.git')
    assert rc == 0, 'git clone FAILED -> turn Internet ON in Settings, then re-run.'
os.chdir('/kaggle/working/PartCrafter')
print('[1/5] pin torch 2.5.1+cu124 ...', flush=True)
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
print('[2/5] setup.sh (repo requirements; numpy downgrade warnings are harmless) ...', flush=True)
!bash settings/setup.sh
print('[3/5] transformers<5 + diffusers 0.38 (hub/tokenizers auto-resolved) ...', flush=True)
!pip install -q "transformers<5" "diffusers==0.38.0"
print('[4/5] build torch_cluster FROM SOURCE (3-8 min, please wait) ...', flush=True)
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
!pip uninstall -y -q torch-cluster
!pip install -q --no-cache-dir --no-build-isolation torch-cluster
print('[5/5] install finished in %d sec' % int(time.time() - t0), flush=True)


In [ ]:
# 2b) Import check (full pipeline incl. torch_cluster). MUST print IMPORTS OK.
import subprocess, sys
r = subprocess.run([sys.executable, '-c',
    'import torch, transformers, diffusers; from torch_cluster import fps; '
    'from src.pipelines.pipeline_partcrafter import PartCrafterPipeline; '
    'print("IMPORTS OK | torch", torch.__version__, "| transformers", transformers.__version__, '
    '"| diffusers", diffusers.__version__, "| cuda", torch.cuda.is_available())'],
    cwd='/kaggle/working/PartCrafter',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/PartCrafter'},
    capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else '')
print('RESULT:', 'OK' if r.returncode == 0 else 'FAILED (paste the lines above)')


In [ ]:
B64 = "..."
# 3) Decode the EMBEDDED input image (no upload needed)
import base64
IMG = '/kaggle/working/PartCrafter/mecha_src.jpg'
open(IMG, 'wb').write(base64.b64decode(B64))
print('wrote', IMG)


In [ ]:
# 4) Generate parts. CAPTURES + SHOWS the inference output so errors are visible.
# num_parts=4 + num_tokens=768 keeps it within T4 memory. Raise num_parts later if it works.
NUM_PARTS = 4
import os, subprocess, sys
os.chdir('/kaggle/working/PartCrafter')
env = dict(os.environ, PYTHONPATH='/kaggle/working/PartCrafter')
cmd = [sys.executable, 'scripts/inference_partcrafter.py',
       '--image_path', '/kaggle/working/PartCrafter/mecha_src.jpg',
       '--num_parts', str(NUM_PARTS), '--tag', 'mecha', '--render',
       '--num_tokens', '768']
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print('===== STDOUT (last 4000) =====')
print(p.stdout[-4000:])
print('===== STDERR (last 4000) =====')
print(p.stderr[-4000:])
print('inference exit code:', p.returncode)


In [ ]:
# 5) Collect the GLB into /kaggle/working (download from the right Output panel)
import glob, os, shutil
glbs = sorted(glob.glob('/kaggle/working/PartCrafter/results/**/*.glb', recursive=True),
              key=os.path.getmtime)
print('GLB files:', glbs)
if glbs:
    out = '/kaggle/working/mecha_parts.glb'
    shutil.copy(glbs[-1], out)
    print('SAVED', out, os.path.getsize(out), 'bytes -> download from the Output panel')
    from IPython.display import FileLink
    display(FileLink('mecha_parts.glb'))
else:
    print('No GLB - check cell 4 output (OOM -> set NUM_PARTS=4 and re-run cell 4).')
